# Data Preparation
This dataset contains comprehensive information on the air quality and its impact on public health for 5,811 records. It includes variables such as air quality index (AQI), concentrations of various pollutants, weather conditions, and health impact metrics. The target variable is the health impact class, which categorizes the health impact based on the air quality and other related factors.

This dataset offers a comprehensive view of the relationship between air quality and public health, making it ideal for research, predictive modeling, and statistical analysis.

In [2]:
include("utils/utils.jl")

    Updating registry at `C:\Users\alons\.julia\registries\General.toml`
   Resolving package versions...
   Installed Zygote ───────────── v0.6.73
   Installed Statistics ───────── v1.11.1
   Installed MLDataDevices ────── v1.5.3
   Installed ForwardDiff ──────── v0.10.38
   Installed OneHotArrays ─────── v0.2.6
   Installed Optimisers ───────── v0.3.4
   Installed Adapt ────────────── v4.1.1
   Installed Flux ─────────────── v0.14.25
   Installed KernelAbstractions ─ v0.9.29
   Installed ChainRules ───────── v1.72.1
    Updating `C:\Users\alons\.julia\environments\v1.11\Project.toml`
  [587475ba] + Flux v0.14.25
    Updating `C:\Users\alons\.julia\environments\v1.11\Manifest.toml`
  [621f4979] + AbstractFFTs v1.5.0
  [7d9f7c33] + Accessors v0.1.38
  [79e6a3ab] + Adapt v4.1.1
  [dce04be8] + ArgCheck v2.3.0
  [a9b6321e] + Atomix v0.1.0
  [198e06fe] + BangBang v0.4.3
  [9718e550] + Baselet v0.1.1
  [fa961155] + CEnum v0.5.0
  [082447d4] + ChainRules v1.72.1
  [d360d2e6] + ChainRulesCore

LoadError: LoadError: ArgumentError: Package ScikitLearn not found in current path.
- Run `import Pkg; Pkg.add("ScikitLearn")` to install the ScikitLearn package.
in expression starting at c:\Users\alons\Desktop\MIA\ML\Labs\ML_Final\ML_Project\utils\utils.jl:8

In [ ]:
# Load the dataset from the 'dataset' folder
data = CSV.read("datasets/air_quality_health_impact_data.csv", DataFrame)

# Check the dataset
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,RecordID,2906.0,1,2906.0,5811,0,Int64
2,AQI,248.438,0.00581738,249.128,499.859,0,Float64
3,PM10,148.655,0.0158481,147.635,299.902,0,Float64
4,PM2_5,100.224,0.0315489,100.506,199.985,0,Float64
5,NO2,102.293,0.00962478,102.988,199.98,0,Float64
6,SO2,49.4568,0.0110232,49.5302,99.9696,0,Float64
7,O3,149.312,0.001661,149.56,299.937,0,Float64
8,Temperature,14.9755,-9.991,14.9424,39.9634,0,Float64
9,Humidity,54.7769,10.0015,54.5439,99.9975,0,Float64


In [10]:
input_data = Matrix(data[!, 1:13]);
output_data = Int.(data[!, 15]);

@assert input_data isa Matrix
@assert output_data isa Vector{Int64}

In [11]:
output_data = oneHotEncoding(vec(output_data))

5811×5 BitMatrix:
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 0  1  0  0  0
 0  1  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 ⋮           
 0  0  1  0  0
 0  0  1  0  0
 0  0  1  0  0
 0  1  0  0  0
 0  0  0  0  1
 0  0  1  0  0
 0  1  0  0  0
 0  0  0  0  1
 1  0  0  0  0

### Which normalization method shoould we use?

In [13]:

(tr_idx, test_idx) = holdOut(size(input_data, 1), 0.2)

train_input = input_data[tr_idx,:]
train_output = output_data[tr_idx]
test_input = input_data[test_idx,:]
test_output = output_data[test_idx]

# First need to generate the parameters from only the train
norm_paramters = calculateZeroMeanNormalizationParameters(train_input)

# normalize the train using the previous parameters
normalizeZeroMean!(train_input, norm_paramters)
# normalize the test using the  train parameters
normalizeZeroMean!(test_input, norm_paramters)


println("DIMENSIONS:")
println("train_input:",size(train_input))
println("train_output:",size(train_output))
println("test_input:",size(test_input))
println("test_output:",size(test_output))

DIMENSIONS:
train_input:(4649, 13)
train_output:(4649,)
test_input:(1162, 13)
test_output:(1162,)


In [14]:
using ScikitLearn

@sk_import svm: SVC
@sk_import tree: DecisionTreeClassifier
@sk_import neighbors: KNeighborsClassifier
@sk_import neural_network: MLPClassifier
@sk_import ensemble:StackingClassifier

PyObject <class 'sklearn.ensemble._stacking.StackingClassifier'>

In [25]:
kFoldIndices = crossvalidation(size(output_data,1), 10)
#Model type for SVM
estimators = [:SVM, :DecisionTree, :KNN, :ANN, :ANN]

# Model hyperparameters specific
modelsHyperParameters = [Dict(
    "kernel"=> "rbf",
    "degree"=> 3,
    "gamma"=> 0.0,
    "C" => 1.0 ),
    
    Dict(
        "max_depth" => 5,
        "random_state" => 42
    ),
    
    Dict(
        "k" => 5 
    ),
    
    Dict(
        "topology" => (100, 50),        
        "maxEpochs" => 200,             
        "learningRate" => 0.001,         
        "validation_fraction" => 0.1
    ),
     
    Dict(
        "topology" => (100, 50),        
        "maxEpochs" => 200,             
        "learningRate" => 0.001         
    )  
]

output_data = collect(reshape(output_data,:,1));
dataset = (input_normal, output_data);

In [26]:
trainClassEnsemble( estimators, modelsHyperParameters, dataset, kFoldIndices)

BoundsError: BoundsError: attempt to access 29055×1 Matrix{Bool} at index [5811-element BitVector, 1:1]